# 🔬 동적 크롤링(Playwright) 데이터 확보 가능 여부 검증

이 노트북은 파이프라인(Pipeline)을 본격적으로 구현하기에 앞서, Playwright를 이용한 **동적 웹 크롤링이 실제로 유효한지(데이터를 정상적으로 긁어올 수 있는지)** 확인하기 위해 작성된 단일 통합 테스트 문서입니다.

아래의 셀들은 실제 실행되었으며, 수집된 결과(Output)가 각 셀 하단에 그대로 보존되어 있습니다.

In [1]:
import asyncio
import json
import urllib.parse

import nest_asyncio
from playwright.async_api import async_playwright

nest_asyncio.apply()

print("✅ 라이브러리 로드 및 비동기 환경 셋업 완료")

✅ 라이브러리 로드 및 비동기 환경 셋업 완료


## 1. Wikipedia 정보상자(Taxobox) 데이터 추출 검증
위키백과 우측의 생물 분류표(Taxonomy)에서 데이터를 정상적으로 파싱할 수 있는지 확인합니다.

In [2]:
async def test_wikipedia_taxonomy(species_name):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        url = f"https://en.wikipedia.org/wiki/{urllib.parse.quote(species_name)}"
        await page.goto(url)
        
        taxonomy = await page.evaluate(r"""
            () => {
                let taxobox = document.querySelector('table.infobox.biota');
                if (!taxobox) return null;
                
                let result = {};
                let rows = taxobox.querySelectorAll('tr');
                rows.forEach(row => {
                    let th = row.querySelector('th');
                    let td = row.querySelector('td');
                    if (th && td && th.innerText.includes(':')) {
                        let key = th.innerText.replace(':', '').trim();
                        let value = td.innerText.trim();
                        result[key] = value;
                    }
                });
                return result;
            }
        """)
        await browser.close()
        return taxonomy

# 불곰(Ursus arctos) 대상 테스트
result_tax = asyncio.run(test_wikipedia_taxonomy("Ursus arctos"))
print("[Wikipedia Taxonomy 파싱 결과]")
print(json.dumps(result_tax, indent=2, ensure_ascii=False))

[Wikipedia Taxonomy 파싱 결과]
{
  "Kingdom": "Animalia",
  "Phylum": "Chordata",
  "Class": "Mammalia",
  "Order": "Carnivora",
  "Family": "Ursidae",
  "Genus": "Ursus",
  "Species": "U. arctos"
}


## 2. Wikipedia 서식지(Habitat/Distribution) 본문 추출 검증
본문 중에서 특정 H2, H3 헤더를 찾아 그 아래의 P태그들만 정확히 긁어올 수 있는지 확인합니다.

In [3]:
async def test_wikipedia_habitat(species_name):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        url = f"https://en.wikipedia.org/wiki/{urllib.parse.quote(species_name)}"
        await page.goto(url)
        
        habitat_text = await page.evaluate(r"""
            () => {
                let headings = Array.from(document.querySelectorAll('h2, h3'));
                for (let h of headings) {
                    let text = h.innerText.toLowerCase();
                    if (text.includes('distribution') || text.includes('habitat') || text.includes('range')) {
                        let node = h.nextElementSibling;
                        let paras = [];
                        while (node && !['H2', 'H3'].includes(node.tagName)) {
                            if (node.tagName === 'P') {
                                paras.push(node.innerText);
                            }
                            node = node.nextElementSibling;
                        }
                        return paras.join('\n').trim();
                    }
                }
                return "";
            }
        """)
        await browser.close()
        return habitat_text

# 뉴트리아(Myocastor coypus) 서식지 텍스트 파싱 테스트
result_hab = asyncio.run(test_wikipedia_habitat("Myocastor coypus"))
print("[Wikipedia Habitat 파싱 결과 (첫 500자)]")
print(result_hab[:500] + "...")

[Wikipedia Habitat 파싱 결과 (첫 500자)]
Native to South America, the coypu has been introduced to North America, Europe, Asia, and Africa, primarily by fur ranchers. The distribution of coypus outside South America tends to contract or expand with successive cold or mild winters. During cold winters, coypus often suffer frostbite on their tails, leading to infection or death. As a result, populations of coypus often contract and even become locally or regionally extinct as in the Scandinavian countries and such US states as Idaho...


## 3. Wikimedia Commons 동적 갤러리 탐색 검증
JS로 렌더링되는 이미지 갤러리(MediaSearch)에서 썸네일을 인식하고 모달을 찾을 수 있는지 확인합니다.

In [4]:
async def test_commons_gallery(keyword):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        url = f"https://commons.wikimedia.org/w/index.php?search={urllib.parse.quote(keyword)}&title=Special:MediaSearch&type=image"
        
        # networkidle 상태까지 대기하여 비동기 이미지가 모두 로드됨을 보장
        await page.goto(url, wait_until="networkidle")
        
        try:
            await page.wait_for_selector("a.sdms-image-result", timeout=5000)
            thumbs = await page.locator("a.sdms-image-result").all()
            print(f"[{keyword}] 갤러리 탐색 성공! 총 {len(thumbs)}개의 이미지 썸네일 노드를 발견했습니다.")
            
            if len(thumbs) > 0:
                # 첫 번째 이미지의 링크(href) 확인 (실제 파이프라인에서는 클릭 이벤트를 발동시킴)
                first_href = await thumbs[0].get_attribute("href")
                print(f"첫 번째 이미지 상대경로: {first_href}")
        except Exception as e:
            print(f"탐색 실패: {e}")
        finally:
            await browser.close()

# 테스트 실행
asyncio.run(test_commons_gallery("Myocastor coypus"))

[Myocastor coypus] 갤러리 탐색 성공! 총 158개의 이미지 썸네일 노드를 발견했습니다.
첫 번째 이미지 상대경로: /wiki/File:Myocastor_coypus_(2).jpg
